# Data selection: 

## Initialization script
Run this code block before running anything else. You only need to do so once.

In [72]:
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets

class Events:
    def __init__(self, event_tree, base_var):
        self.event_tree = event_tree    # The raw event tree from the root file
        self.base_var = base_var        # The base variable to be plotted. This will be 'cut' by other variables
        self._variables = []            # List of strings; the names of variables which will be used
        self.df = []                    # Dataframe of the data from selected variables
        self.original_df = []           # An original copy of the dataframe to be set in create_dataframe

        self._variables.append(base_var)

    @property
    def variables(self):
        return self._variables
    
    @variables.setter
    def variables(self, vars=[]):
        '''Takes in a list of cutter variables and appends them to variables'''
        self.variables.extend(vars)

    def create_dataframe(self):
        '''Creates a pandas dataframe from the list of variables'''
        self.df = self.event_tree.arrays(self.variables, library="pd")
        self.original_df = self.df

    def filter_rows_range(self, var, min, max):
        '''Takes in a variable name string and a min/max and removes any rows where that variable's value does not fall within the range'''
        if var not in self._variables:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")

        self.df = self.df.query(f"{min} < {var} < {max}")

    def filter_rows_bool(self, var, boolean):
        '''Takes in a variable name string and a true/false and removes any rows where that variable's value does not match the boolean'''
        if var not in self._variables:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")
        elif not isinstance(boolean, bool):
            raise TypeError("Error: input must be a boolean")
        
        self.df = self.df.query(f"{var} == {boolean}")
        
    def is_bool_variable(self, var):
        '''Checks whether or not a cutter variable contains boolean data. If not, it will always contain float data instead.'''
        if var not in self._variables:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")
        
        for x in self.df[var].head(100):    # Checks the variable's first 100 entries to see if any of them are non-boolean. 100 might be overkill but whatever
            if not (x == 1 or x == 0):
                return False
        
        return True


class PlotHelper:
    def __init__(self, events):
        self.events = events        # Takes in an Events object so PlotHelper can access its data

    def draw_plots(self):
        '''Placeholder function, all of this will be replaced'''
        arr_before = self.events.original_df[self.events.base_var]
        arr_after = self.events.df[self.events.base_var]

        plt.subplots(figsize=(15, 5))
        plt.subplot(2,2,1)
        plt_before = plt.hist(arr_before, bins=500)

        plt.subplot(2,2,2)
        plt_after = plt.hist(arr_after, bins=500)

        plt.subplot(2,2,3)
        plt_before = plt.hist(arr_before, bins=500)
        plt_after = plt.hist(arr_after, bins=500)

        plt.ylim(0,25000)

        plt.show()
                
                

## Data cutting

In [ ]:
file = uproot.open('data/00385270_00000001_1.dvntuple.root')

event_tree = file[file.keys()[1]]

events = Events(event_tree, "B0_MM")            # Create an events object with base variable B0_MM
plots = PlotHelper(events)

events.variables = ["B0_L0Global_TOS", "B0_P"]
events.create_dataframe()

events.filter_rows_range("B0_P", 35000, 300000)
events.filter_rows_bool("B0_L0Global_TOS", True)

def widget_update(func, var, min, max):
    func(var, min, max)

    plots.draw_plots()

widgets.interact(widget_update, var="B0_P", min=35000, max=300000, func=widgets.fixed(events.filter_rows_range), continuous_update=False)


#plots.draw_plots()


# ok gonna have to change my approach back in the last file, instead of fully removing entries the filters should instead mark entries as unusable and that unusable filter should be applied onto the original dataframe each time the graph is rendered











interactive(children=(Text(value='B0_P', description='var'), IntSlider(value=35000, description='min', max=105…

<function __main__.widget_update(func, var, min, max)>